# All-Solid Rod-Lattice PBG Fiber

```{index} PBG; rod lattice
```
```{index} ARROW guidance
```

The bandgap fiber of the [previous notebook](./4_1_pbg.ipynb) confines
light with a lattice of high-index strands. A bandgap can also be opened by
a lattice of *high-index solid rods* embedded in a lower-index
background, with a defect core of the *same* index as the background.
There is no index contrast to guide the core mode at all, unlike
ordinary step-index fibers. Confinement is instead an antiresonant
effect of the rod lattice, closely related to the ARROW (antiresonant
reflecting optical waveguide) mechanism seen for the [ARF
notebook](./3_1_arf.ipynb). This design and its resonance structure
are studied in [[1]](#references), and the same
fiber is used as a benchmark example in [[2]](#references).

The `rod` dictionary in `fibermode.pbg.fiber_dicts` describes a single
ring of 6 dielectric rods ($n_\text{tube}=1.8$) around a solid core
of index equal to the cladding ($n_\text{core}=n_\text{clad}=1.44$),
at a wavelength of 825 nm.

In [ ]:
import ngsolve as ng
import numpy as np
from ngsolve.webgui import Draw
from fibermode import PBG
from fibermode.pbg.fiber_dicts.rod import params

## Constructing the fiber

In [ ]:
A = PBG(params)
Draw(A.mesh);

In [ ]:
Draw(A.N, A.mesh, 'index', settings={"Objects": {"Wireframe": False, "Edges":False}});
A.n_tube, A.n_clad, A.n_core

## Fundamental mode: scalar search

As with earlier notebooks, we can search for the fundamental mode
using the scalar (frequency-dependent PML) formulation. Because
$\Delta n = n_\text{tube}-n_\text{clad}=0.36$ is fairly large here,
the scalar Helmholtz problem is only a weakly-guiding *approximation*
-- useful for locating roughly where the true (vector) mode sits, but
not itself the physically correct eigenvalue. The search center below
was located by prior numerical experience with this fiber; see
also the tip on obtaining a principled guess in the [previous notebook](4_1_pbg.ipynb).

In [ ]:
Z, y, yl, beta, P, extras = A.leakymode(
    2,
    rad=.01,
    ctr=1.73806037 - 0.01388821j,
    alpha=5,
    npts=2,
    nspan=2,
    niterations=100,
    nrestarts=0,
)

We should, in principle increase the PML radius to avoid the last warning, 
but since this scalar search is only a rough guide to the true vector mode, we  ignore it for now.

In [ ]:
print('Z    =', Z)
print('beta =', beta)
print('CL [dB/m] =', 20 * beta.imag / np.log(10))

## Fundamental mode: vector search

Because the scalar approximation isn't physically trustworthy for
this fiber, we solve the full vector (Maxwell) problem next, the
physically correct target, as in [[2]](#references). We center the search at $(Z_\text{scalar})^2$
anyway, since that's close enough for FEAST to converge onto the true
vector eigenvalues from there; the resulting $Z^2$ does *not* match
$(Z_\text{scalar})^2$, which is expected given the large index
contrast (unlike the ARF notebook, where scalar and vector
formulations of the same near-lossless mode agreed closely).

```{index} near-degenerate; eigenvectors
```
```{index} near-degenerate; adjusting eta_tol
```

The fundamental mode here is a *near-degenerate pair*. The  two
polarization states of the fundamental hybrid mode are split only
slightly by the hexagonal lattice's imperfect rotational symmetry.
Resolving both reliably needs `nspan` comfortably larger than the true
multiplicity of 2 (since with `nspan` set to exactly 2, FEAST's 
subspace-cleaning step can, depending on run-to-run roundoff, 
collapse the subspace to a single vector partway through, after which
it can't converge cleanly to either eigenvalue). So we set `nspan=4`
to give it enough slack to separate the two reliably and adjust `eta_tol`.

In [ ]:
center2 = Z**2   # for the Z computed above, X^2 = (1.73806037 - 0.01388821j) ** 2

betas, Zsqrs, Es, phis, R = A.leakyvecmodes(
    p=4,
    rad=0.1,
    ctr=center2,
    alpha=A.alpha,
    npts=2,
    nspan=4,
    niterations=100,
    nrestarts=0,
    eta_tol=1e-15  # need this to prevent accidental truncation of the near-degenerate pair
)

In [ ]:
print('Z^2 =', Zsqrs)
print('CL [dB/m]:', 20 * np.array(betas).imag / np.log(10))

`leakyvecmodes` returns the transverse electric fields `Es` (in the
H(curl) space) and the scaled longitudinal components `phis` (in H1),
one pair per polarization state:

In [ ]:
for e in Es:
    Draw(e.real, A.mesh, vectors={'grid_size': 40}, 
         settings={"Objects": {"Wireframe": False}})

In [ ]:
for phi in phis:
    Draw(phi, A.mesh, settings={"Objects": {"Wireframe": False}})

Both polarization states show the same antiresonant confinement
mechanism: the mode is trapped in the solid core purely by the rod
lattice's bandgap effect, with no index contrast between core and
cladding to rely on.

<a id='references'></a>
## References

[1] N. M. Litchinitser, S. C. Dunn, B. Usner, B. J. Eggleton, T. P. White, R. C. McPhedran, and C. M. de Sterke, "Resonances in microstructured optical waveguides," *Optics Express* 11(11), 1243-1251 (2003). DOI: [10.1364/OE.11.001243](https://doi.org/10.1364/OE.11.001243)

[2] J. Gopalakrishnan, J. Grosek, G. Pinochet-Soto, and P. Vandenberge, "Adaptive resolution of fine scales in modes of microstructured optical fibers," *SIAM Journal on Scientific Computing*, 47(1):B108-B130, 2025. Open Access, DOI: [10.1137/24M1651605](https://doi.org/10.1137/24M1651605)